# X02 In-context Learning and Induction Heads


## Goal

Studies induction as a concrete circuit object and is a classic entry point to why transformers can copy patterns and support in-context behavior.


## Why this paper matters now

It connects attention reading, circuit language, and a concrete behavioral phenomenon, making it a first stop for classic transformer-circuits work.


## Notebook and deliverable

- Source: https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html
- Notebook: `notebooks/extensions/en/x02_induction_heads.ipynb`
- Prereqs: X01, M06
- Reproduce one minimal copying task in Colab and explain why an induction head is more like a mechanism than a single attention hotspot.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/circuits-zoom-in-fresh-20260325.git"
REPO_DIR = "circuits-zoom-in-fresh-20260325"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import math
import matplotlib.pyplot as plt
import numpy as np

tokens = ["A", "B", "C", "A", "B", "?"]
prev_tokens = ["<bos>", "A", "B", "C", "A", "B"]
vocab = {"<bos>": [0, 0, 0], "A": [1, 0, 0], "B": [0, 1, 0], "C": [0, 0, 1]}
keys = np.array([vocab[token] for token in prev_tokens[:-1]], dtype=float)
values = np.array([vocab[token] for token in tokens[:-1]], dtype=float)
query = np.array(vocab[prev_tokens[-1]], dtype=float)

scores = 5.0 * (keys @ query) / math.sqrt(query.shape[0])
weights = np.exp(scores - scores.max())
weights = weights / weights.sum()
prediction = weights @ values
winner = ["A", "B", "C"][int(np.argmax(prediction))]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(range(len(weights)), weights, color="#1f5f8b")
ax.set_xticks(range(len(weights)), [f"pos {i}:{token}" for i, token in enumerate(tokens[:-1])], rotation=20)
ax.set_ylabel("attention weight")
ax.set_title("Final query attends to earlier positions")
plt.tight_layout()

print("Final previous-token query:", prev_tokens[-1])
print("Predicted next-token vector:", np.round(prediction, 3))
print("Recovered token:", winner)


## Takeaway

An induction pattern is more than a bright attention edge. It is a copy mechanism anchored to repeated context.


## Colab-first replication workflow


### Before you run

- Write one prediction about the mechanism before you run the cells.
- Name the baseline you are comparing against.
- Decide what result would count as a failure of your favorite story.


### After you run

- Separate observation from inference in your notes.
- Mark one ambiguity that still remains after the reproduction.
- Write one next experiment that would reduce that ambiguity.


### Ship these outputs

- Reproduce one minimal copying task in Colab and explain why an induction head is more like a mechanism than a single attention hotspot.
- One experiment log with the exact settings you changed.
- One paragraph called 'what this reproduction still does not prove'.


## Self-check questions

- Why does 'one position attends backward' still not imply an induction mechanism?
- In your toy copying task, how does performance change as the repeat distance changes?
- What does an induction head lose if no earlier head provides a matching cue?
- If you cannot answer at least two questions without reopening the notebook, rerun the reproduction and rewrite the memo.
